In [ ]:
import json
import logging
import requests
from time import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(), logging.FileHandler("portfolio.log")]
)



def execution_timer(func):
    def wrapper(*args, **kwargs):
        start_time = time()
        result = func(*args, **kwargs)
        end_time = time()
        logging.info(f"Execution of '{func.__name__}' took {end_time - start_time:.2f} seconds.")
        return result
    return wrapper

class ReportSaver:
    def __init__(self, filename):
        self.filename = filename
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, 'w')
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        logging.info("Report file successfully closed by the Context Manager.")
        return False

class Asset:
    def __init__(self, name, symbol, quantity):
        self.name = name
        self.symbol = symbol
        self._quantity = quantity

    @property
    def quantity(self):
        return self._quantity

    def get_value(self, current_price):
        raise NotImplementedError("Subclasses must implement get_value()!")

class CryptoAsset(Asset):
    def __init__(self, name, symbol, quantity, api_id):
        super().__init__(name, symbol, quantity)
        self.api_id = api_id

    def get_value(self, current_price):
        return self._quantity * current_price

class Portfolio:
    def __init__(self):
        self.assets = []

    def add_asset(self, asset):
        self.assets.append(asset)

    def __iter__(self):
        self._index = 0
        return self

    def __next__(self):
        if self._index < len(self.assets):
            current_asset = self.assets[self._index]
            self._index += 1
            return current_asset
        raise StopIteration

def summary_generator(portfolio, price_data):
    for asset in portfolio:
        price = price_data.get(asset.api_id, {}).get('usd', 0)
        value = asset.get_value(price)
        yield f"{asset.name} ({asset.symbol}): Qty {asset.quantity} | Price: ${price:,.2f} | Value: ${value:,.2f}"

@execution_timer
def fetch_live_prices(crypto_ids):
    logging.info("Connecting to CoinGecko API to fetch live prices...")
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": ",".join(crypto_ids),
        "vs_currencies": "usd"
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as error:
        logging.error(f"API Request failed: {error}")
        return {}

if __name__ == "__main__":
    bitcoin = CryptoAsset("Bitcoin", "BTC", 0.5, "bitcoin")
    ethereum = CryptoAsset("Ethereum", "ETH", 2.5, "ethereum")

    my_portfolio = Portfolio()
    my_portfolio.add_asset(bitcoin)
    my_portfolio.add_asset(ethereum)

    crypto_keys = [asset.api_id for asset in my_portfolio]
    live_prices = fetch_live_prices(crypto_keys)

    report_data = {
        "portfolio_summary": [],
        "total_value": 0.0
    }

    print("\n--- Live Portfolio Tracking ---")
    
    for line in summary_generator(my_portfolio, live_prices):
        print(line)
        report_data["portfolio_summary"].append(line)

    total = sum(
        asset.get_value(live_prices.get(asset.api_id, {}).get('usd', 0))
        for asset in my_portfolio
    )
    report_data["total_value"] = total
    print(f"Total Portfolio Value: ${total:,.2f}\n")

    with ReportSaver("portfolio_report.json") as file:
        json.dump(report_data, file, indent=4)

2026-07-14 18:37:21,084 - INFO - Connecting to CoinGecko API to fetch live prices...
2026-07-14 18:37:21,986 - INFO - Execution of 'fetch_live_prices' took 0.90 seconds.
2026-07-14 18:37:21,991 - INFO - Report file successfully closed by the Context Manager.



--- Live Portfolio Tracking ---
Bitcoin (BTC): Qty 0.5 | Price: $63,778.00 | Value: $31,889.00
Ethereum (ETH): Qty 2.5 | Price: $1,859.98 | Value: $4,649.95
Total Portfolio Value: $36,538.95



 METHOD OVERLOADING 

In [2]:
class D:
    def add(self, a, b):
        return a + b
    def add(self,a,b,c):
        return a+b+c
d=D()
print(d.add(1,2,3))
print(d.add(1,2))

6


TypeError: D.add() missing 1 required positional argument: 'c'

In [4]:
pip install multipledispatch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import multipledispatch
class D:
    @multipledispatch.dispatch(int, int)
    def add(self, a, b):
        return a + b
    @multipledispatch.dispatch(int, int, int)
    def add(self,a,b,c):
        return a+b+c
    @multipledispatch.dispatch(str,str)
    def add(self, a, b):
        return a + b
d=D()
print(d.add("a","b"))
print(d.add(1,2,3))
print(d.add(1,2))

ab
6
3


OVERRIDING

In [ ]:
class parent:
    def method(self):
        print("This is parent")
class child(parent):
    def method(self):
        print("This is child")
c=child()
c.method()


This is child


In [ ]:
class parent1:
    def method(self):
        print("This is parent1")
class parent2:
    def method(self):
        print("This is parent2")
class child(parent1,parent2):
    def method(self):
        print("This is child")
c=child()
c.method() 


This is child


In [6]:
class parent1:
    def method(self):
        print("This is parent1")
class parent2:
    def method(self):
        print("This is parent2")
class child(parent2, parent1):
    pass
c=child()
c.method() 


This is parent2


In [10]:
class parent:
    def __init__(self):
        print("This is parent")
class child(parent):
    def __init__(self):
        print("This is child")
    def method(self):
        print("This is child2")
c=child()


This is child


ACCESSING PARENT METHOD

In [12]:
class parent:
    def method(self):
        print("This is parent")
class child(parent):
    def method(self):
        parent.method(self)
        print("This is child2")
c=child()
c.method()


This is parent
This is child2


In [13]:
class parent:
    def method(self):
        print("This is parent")
class child(parent):
    def method(self):
        super().method()
        print("This is child2")
c=child()
c.method()


This is parent
This is child2


POLYMORPHISM

In [15]:
def sum(*args):
    if args:
        start=type(args[0])()
        for i in args:
            start+=i
        return start
print(sum(1,2))
print(sum(3.2,5.4))
print(sum("a","b"))
print(sum([1,2,3],[4,5,6]))

3
8.600000000000001
ab
[1, 2, 3, 4, 5, 6]


In [16]:
class A:
    def fun(self):
        print("class A")
class B:
    def fun(self):
        print("class B")
class C:
    def fun(self):
        print("class C")
def poly(obj):
    obj.fun()

obj1=A()
obj2=B()
obj3=C()
poly(obj1)
poly(obj2)
poly(obj3)


class A
class B
class C


POLYMORPHISM USING OVERLOADING

In [24]:
from multipledispatch import dispatch
class A:
    @dispatch(int,int)
    def add(self,a,b):
        return a+b
    @dispatch(float,float,float)
    def add(self,a,b,c):
        return a+b+c
    @dispatch(str,str)
    def add(self,a,b):
        return a+b   

obj=A()
print(obj.add(2,3))
print(obj.add(2.1,4.2,3.2))
print(obj.add("a","b"))            

5
9.5
ab


#ITERATOR

In [28]:
import random
class Dice:
    def __init__(self, rolls):
        self.rolls=rolls
        self.count=0
    def __iter__(self):
        return self
    def __next__(self):
        if self.count < self.rolls:
            self.count+=1
            return random.randint(1,6)
        else:
            raise StopIteration
d=Dice(0)
for i in d:
    print(i)

Generator ==> Function that behaves like a iterator (it can be used in a for loop)
            passes a function, return value, then resumes
            uses yield, instead of return 
            iterate without loading everything in memory (ex reading large files )
            return = pouring bucket
            yield= drip faucet

In [15]:
# def count_num(n):
#     numbers=[]
#     count=1
#     while count <= n:
#         numbers.append(count)
#         count+=1
#     return numbers
# number=int(input("enter the number"))
# for n in count_num(number):
#     print(n)

#generator version
def count_num(n):
    count=1
    while count <= n:
        yield count
        count+=1
number=int(input("enter the number"))
for n in count_num(number):
    print(n)


1
2
3
4
5
6
7
8
9
10


In [17]:
def read_file(filename):
    with open(filename) as file:
        for line in file:
            yield line.strip()
filename='test.txt'
for line in read_file(filename):
    print(line)


A decorator feature in Python wraps in a function, appends several functionalities to existing code and then returns it. Methods and functions are known to be callable as they can be called. Therefore, a decorator is also a callable that returns callable. This is also known as metaprogramming as at compile time a section of program alters another section of the program. Note: For more information, refer to Decorators in Python

Python @property decorator
@property decorator is a built-in decorator in Python which is helpful in defining the properties effortlessly without manually calling the inbuilt function property(). Which is used to return the property attributes of a class from the stated getter, setter and deleter as parameters. Now, lets see some examples to illustrate the use of @property decorator in Python: Example


ITERATOR

In [19]:
import random
class Dice:
    def __init__(self,rolls):
        self.rolls=rolls
        self.count=0
    def __iter__(self):
        return self
    def __next__(self):
        if self.count<self.rolls:
            self.count+=1
            return random.randint(1,6)
        else:
            raise StopIteration
# dice=Dice(3)
# for i in dice:
#     print(i)
dice=[die for die in Dice(3)]
print(dice)

# internally works like this

# iterator=iter(dice)
# while True:
#     try:
#         roll=next(iterator)
#         print(roll)
#     except StopIteration
#     break 
        

[3, 6, 3]


ABSTRACT CLASS

In [ ]:
from abc import ABC, abstractmethod
class A(ABC):
    @abstractmethod
    def method1(self):
      pass
    def method2(self):
  print("ehhh concreate")
    @abstractmethod  
    def method3(self):
  pass
    
class B(A):
    def method1(self):
      print("method1 Implemented in subclass")
    def method3(self):
  print("Method2 is Implemented in sub class")
a=B()
a.method1()
a.method3()

SyntaxError: invalid non-printable character U+00A0 (3017260571.py, line 3)

DECORATOR

In [12]:
def add_sprinkle(func):
    def wrapper(*args,**kwargs):
        print("YOU add sprinkle")
        func(*args,**kwargs)
    return wrapper
def add_fudge(func):
    def wrapper(*args,**kwargs):
        print("YOU add fudge")
        func(*args,**kwargs)
    return wrapper
@add_sprinkle
@add_fudge
def get_icecream(flavour):
    print(f"ICE CREAM is {flavour} ")
get_icecream('vanilla')

YOU add sprinkle
YOU add fudge
ICE CREAM is vanilla 


In [20]:
import json
a ={"name":"John",
   "age":31,
    "Salary":25000}
json_string=json.dumps(a)
print(json_string)

{"name": "John", "age": 31, "Salary": 25000}


In [21]:
# Polymorphism = Greek word that means to "have many forms or faces"
#                               Poly = Many
#                               Morphe = Form

#                TWO WAYS TO ACHIEVE POLYMORPHISM
#                1. Inheritance = An object could be treated of the same type as a parent class
#                2. "Duck typing" = Object must have necessary attributes/methods

from abc import ABC, abstractmethod

class Shape(ABC):

    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return self.base * self.height * 0.5

class Pizza(Circle):
    def __init__(self, topping, radius):
        super().__init__(radius)
        self.topping = topping

shapes = [Circle(4), Square(5), Triangle(6, 7), Pizza("pepperoni", 15)]

for shape in shapes:
    print(f"{shape.area()}cm²")

50.24cm²
25cm²
21.0cm²
706.5cm²
